# Pasture Studies — bounded Kaggle training and readiness

`PREFLIGHT`, `SMOKE`, and `READINESS` remain diagnostic-only. Earlier exploratory modes preserve their fixed budgets. `AUTORESEARCH_BATCH_1/2` contain 12 nonformal recovery jobs. `AUTORESEARCH_BATCH_3` adds six K/L jobs using a separately versioned outcome-aware training target after the first cohorts exposed action-intent reward exploitation. Each mode is authorized through an exact immutable config and release binding. All selection uses new common worlds and the original simulator reward; reserved data, formal modes, and locked outcomes remain unavailable.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, time, uuid, zipfile

REPO_URL = "https://github.com/santleme/pasture-studies.git"
# Updated to the reviewed immutable source commit before publication.
PASTURE_DEPLOYMENT = None  # KAGGLE_DEPLOYMENT_CONFIG_MARKER
DEPLOY = PASTURE_DEPLOYMENT
if not isinstance(DEPLOY, dict):
    raise RuntimeError("Notebook was not prepared by scripts/kaggle_deploy.py")
MODE = DEPLOY.get("mode", "")
if MODE in {"EXPERIMENT", "FULL_VALIDATION"}:
    raise RuntimeError("Experimental execution is hard-disabled in this launcher")
if MODE not in {"PREFLIGHT", "SMOKE", "READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON", "EXPLORATORY_ITERATION_2", "AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"}:
    raise RuntimeError("Unsupported execution mode")
EXPECTED_COMMIT = DEPLOY.get("source_commit")
if not isinstance(EXPECTED_COMMIT, str) or len(EXPECTED_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in EXPECTED_COMMIT):
    raise RuntimeError("Invalid immutable source commit pin")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RUN_ID = f"{MODE.lower()}-{uuid.uuid4().hex}"
ARTIFACT_ROOT = WORK_ROOT / f"pasture-release-{RUN_ID}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=False)

def run(cmd, *, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True, text=True)

print("Mode:", MODE, "run_id:", RUN_ID, "source SHA:", EXPECTED_COMMIT)
print("Python:", sys.version, "platform:", platform.platform())


## Immutable source checkout and detached release verification

The host-side release manifest binds committed source notebook bytes and the exact injected staged notebook bytes. Kaggle receives only the notebook code file; the deployment config is embedded as a literal and the external manifest remains a detached host attestation. The launcher verifies the pinned source commit and runtime profile.


In [ ]:
REPO_DIR = WORK_ROOT / f"pasture-source-{EXPECTED_COMMIT[:12]}"
if not REPO_DIR.exists():
    # Partial clone plus sparse checkout keeps reserved evaluation artifacts
    # out of the Kaggle runtime; only immutable runtime code is materialized.
    run(["git", "clone", "--filter=blob:none", "--no-checkout", "--no-tags", REPO_URL, REPO_DIR])
run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "--filter=blob:none", "origin", EXPECTED_COMMIT])
run(["git", "-C", REPO_DIR, "sparse-checkout", "init", "--no-cone"])
run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone",
     "/farm_sim/", "/validation/train_torch_mvp1.py", "/validation/profile_torch_training.py",
     "/validation/evaluate_torch_mvp1.py", "/validation/compare_exploratory_checkpoints.py",
     "/validation/run_autoresearch_training.py",
     "/scripts/kaggle_release.py",
     "/requirements-training.txt", "/notebooks/pasture_studies_kaggle.ipynb"])
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])
resolved = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if resolved != EXPECTED_COMMIT:
    raise RuntimeError("Pinned source commit mismatch")
if (REPO_DIR / "artifacts").exists() or (REPO_DIR / "validation/kaggle-mvp1").exists():
    raise RuntimeError("Sparse source checkout unexpectedly includes reserved evaluation artifacts")
source_notebook = REPO_DIR / "notebooks/pasture_studies_kaggle.ipynb"
source_notebook_sha = hashlib.sha256(source_notebook.read_bytes()).hexdigest()

os.chdir(REPO_DIR)
import numpy as np
import torch
from importlib.metadata import version

runtime = {"python": platform.python_version(), "numpy": np.__version__,
           "pytorch": torch.__version__, "cuda_runtime": torch.version.cuda,
           "cuda_available": bool(torch.cuda.is_available()),
           "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
           "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []}
print("Observed runtime:", json.dumps(runtime, sort_keys=True))
if source_notebook_sha != DEPLOY.get("source_notebook_sha256"):
    raise RuntimeError("Committed source notebook SHA differs from staged deployment binding")
if MODE in {"READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON", "EXPLORATORY_ITERATION_2", "AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"}:
    if runtime != DEPLOY.get("runtime_profile"):
        raise RuntimeError("Observed runtime differs from the staged T4x2 profile")
    if MODE == "READINESS":
        from scripts.kaggle_release import authorize_readiness
        authorize_readiness(DEPLOY.get("readiness_config"))
        if DEPLOY.get("readiness_config", {}).get("locked_test_access") is not False:
            raise RuntimeError("Readiness config must deny locked outcome access")
    else:
        from scripts.kaggle_release import authorize_mode
        authorize_mode(MODE, {"release_manifest_sha256": DEPLOY.get("release_manifest_sha256")},
                       config=DEPLOY.get("exploratory_config"))
        if DEPLOY.get("exploratory_config", {}).get("locked_test_access") is not False:
            raise RuntimeError("Exploratory config must deny locked outcome access")

# Record exact versions of the historical pins for context; READINESS never installs or changes them.
requirements = REPO_DIR / "requirements-training.txt"
print("Historical scientific dependency pins (unchanged):\n", requirements.read_text())


## Runtime preflight and CUDA exercise

Exploratory batches verify the staged CUDA T4x2 runtime and a real forward/backward operation. Bounded calibration compares CPU, both individual GPUs, and concurrent independent GPU workers before selecting a measured execution schedule. No GPU-acceleration claim is made without these measurements.


In [ ]:
if MODE in {"READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON", "EXPLORATORY_ITERATION_2", "AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"}:
    if not DEPLOY.get("gpu_required") or not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("READINESS requires exactly two visible CUDA GPUs")
    names = runtime["gpu_names"]
    if any("T4" not in name for name in names):
        raise RuntimeError(f"Expected T4x2 hardware, found {names}")
    from farm_sim.torch_training import ActorCritic
    model = ActorCritic(value_head=True, seed=37331, dtype=torch.float32).to("cuda:0")
    x = torch.zeros((4, 20), dtype=torch.float32, device="cuda:0")
    logits, values = model(x)
    (logits.square().mean() + values.square().mean()).backward()
    if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()):
        raise RuntimeError("CUDA forward/backward gradient check failed")
    print("CUDA forward/backward: PASS", names)
elif MODE == "PREFLIGHT":
    print("PREFLIGHT complete; no training launched")
elif MODE == "SMOKE":
    if DEPLOY.get("gpu_required") and not torch.cuda.is_available():
        raise RuntimeError("GPU-required SMOKE cannot fall back to CPU")
    smoke_root = ARTIFACT_ROOT / "smoke"
    jobs = [("cpu", "reinforce", "cpu"), ("cpu", "a2c", "cpu")]
    if torch.cuda.is_available(): jobs.append(("cuda", "a2c", "cuda"))
    for condition, algorithm, device in jobs:
        output = smoke_root / condition / algorithm
        run([sys.executable, "-m", "validation.train_torch_mvp1", "--smoke-update",
             "--algorithm", algorithm, "--device", device, "--seed", "37331",
             "--output", output, "--metrics", output / "metrics.jsonl"], cwd=REPO_DIR)
        latest = output / "smoke" / algorithm / "LATEST"
        if not latest.is_file(): raise RuntimeError(f"SMOKE checkpoint missing: {latest}")
        generations = list((latest.parent).glob("generation-*/manifest.json"))
        if len(generations) != 1: raise RuntimeError(f"SMOKE expected one fresh generation: {latest.parent}")
    print("SMOKE pipeline diagnostics completed; no experiment/evaluation was run")


## Two-GPU bounded DDP and interrupted-run recovery

This runs a two-update A2C recovery diagnostic, not an experiment. The first invocation commits one update and checkpoint; the second resumes from `LATEST` and commits the second. Checkpoints must contain readiness-only flags; rank records must show both CUDA devices doing nonzero finite-gradient updates with synchronized model digests. No policy evaluation is run.


In [ ]:
if MODE == "READINESS":
    readiness_root = ARTIFACT_ROOT / "readiness-ddp"
    readiness_root.mkdir()
    diagnostic_seed = 37331  # reserved pipeline diagnostic; not a preregistration split seed
    base = [sys.executable, "-m", "torch.distributed.run", "--standalone",
            "--nproc_per_node=2", "-m", "validation.train_torch_mvp1",
            "--readiness-ddp", "--algorithm", "a2c", "--device", "cuda",
            "--seed", str(diagnostic_seed), "--episodes", "32", "--checkpoint-every", "1",
            "--output", str(readiness_root)]
    started = time.time()
    run(base + ["--max-updates", "1"], cwd=REPO_DIR)
    latest = readiness_root / "readiness-only/a2c/LATEST"
    if not latest.is_file(): raise RuntimeError("First readiness checkpoint was not committed")
    first_generation = latest.read_text().strip()
    run(base + ["--resume", "latest", "--max-updates", "1"], cwd=REPO_DIR)
    evidence_path = readiness_root / "readiness-only/a2c/readiness-evidence.json"
    evidence = json.loads(evidence_path.read_text())
    if evidence.get("benchmark_eligible") is not False or evidence.get("evaluation_performed") is not False or evidence.get("locked_split_accessed") is not False:
        raise RuntimeError("Readiness evidence has unsafe eligibility/access flags")
    updates = evidence.get("updates", [])
    if len(updates) != 2 or [row.get("update") for row in updates] != [1, 2]:
        raise RuntimeError("Interrupted-run recovery did not produce exactly two consecutive updates")
    rank_rows = [record for update in updates for record in update.get("episode_ids_by_rank", [])]
    if {row.get("cuda_device_index") for row in rank_rows} != {0, 1}:
        raise RuntimeError("Both T4 GPUs did not report a readiness update")
    if not all(row.get("gradient_finite") and row.get("optimizer_step_changed_model") for row in rank_rows):
        raise RuntimeError("At least one GPU rank did not perform finite-gradient training work")
    if len({update.get("model_digest") for update in updates}) != 2:
        raise RuntimeError("Expected model state to change between optimizer updates")
    checkpoint = readiness_root / "readiness-only/a2c"
    generations = sorted(checkpoint.glob("generation-*/manifest.json"))
    if len(generations) != 2:
        raise RuntimeError("Expected exactly two committed checkpoint generations after recovery")
    from farm_sim.torch_checkpoint import _is_complete_generation
    for path in generations:
        manifest = json.loads(path.read_text())
        if not _is_complete_generation(path.parent, expected_algorithm="a2c"):
            raise RuntimeError(f"Checkpoint generation failed integrity validation: {path.parent}")
        payload = torch.load(path.parent / "training.pt", map_location="cpu", weights_only=True)
        state = payload.get("state", {})
        if (manifest.get("config", {}).get("protocol") != "readiness-ddp-v1"
                or state.get("readiness_only") is not True
                or state.get("benchmark_eligible") is not False):
            raise RuntimeError("Checkpoint is not marked readiness-only")
    result = {"mode": MODE, "run_id": RUN_ID, "source_commit": EXPECTED_COMMIT,
              "release_manifest_sha256": DEPLOY.get("release_manifest_sha256") if MODE == "READINESS" else None,
              "notebook_sha256": source_notebook_sha, "runtime": runtime,
              "diagnostic_seed": diagnostic_seed, "updates": 2, "resumed_from_generation": first_generation,
              "checkpoint_generations": [p.parent.name for p in generations],
              "gpu_rank_indices": sorted({r["cuda_device_index"] for r in rank_rows}),
              "cuda_forward_backward": "PASS", "locked_outcomes_accessed": False,
              "evaluation_performed": False, "benchmark_eligible": False,
              "elapsed_seconds_diagnostic_only": time.time() - started}
    (ARTIFACT_ROOT / "readiness-report.json").write_text(json.dumps(result, indent=2, sort_keys=True) + "\n")
    print(json.dumps(result, indent=2, sort_keys=True))

elif MODE == "EXPLORATORY":
    # Fixed, deliberately modest learning run. It does not call formal evaluators
    # or read any registered split; five generated worlds are descriptive only.
    exploratory_root = ARTIFACT_ROOT / "exploratory-only"
    exploratory_root.mkdir()
    cfg = DEPLOY["exploratory_config"]
    if cfg["episodes"] != 256 or cfg["optimizer_updates"] != 16 or cfg["device_count"] != 1:
        raise RuntimeError("Exploratory budget differs from the fixed nonformal config")
    command = [sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-a2c",
               "--algorithm", "a2c", "--device", "cuda", "--seed", str(cfg["seed"]),
               "--episodes", str(cfg["episodes"]), "--eval-seeds", str(cfg["evaluation_episodes"]),
               "--checkpoint-every", "1", "--output", str(exploratory_root)]
    started = time.time()
    run(command, cwd=REPO_DIR)
    learner_dir = exploratory_root / "exploratory-only" / "a2c"
    # The trainer writes its shared update log at args.output/metrics.jsonl,
    # one level above the per-algorithm checkpoint directory.
    metrics_path = exploratory_root / "exploratory-only" / "metrics.jsonl"
    rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    if len(rows) != 16 or [row.get("update") for row in rows] != list(range(1, 17)):
        raise RuntimeError("Exploratory A2C did not complete its fixed 16 updates")
    if any(row.get("benchmark_eligible") is not False or row.get("formal_eligible") is not False
           or row.get("exploratory_only") is not True for row in rows):
        raise RuntimeError("Exploratory metrics are not safely marked nonformal")
    latest = learner_dir / "LATEST"
    if not latest.is_file(): raise RuntimeError("Exploratory A2C checkpoint pointer is missing")
    from farm_sim.torch_checkpoint import _is_complete_generation
    generations = sorted(learner_dir.glob("generation-*/manifest.json"))
    if len(generations) != 16:
        raise RuntimeError("Expected one integrity-checked checkpoint generation per exploratory update")
    for manifest_path in generations:
        if not _is_complete_generation(manifest_path.parent, expected_algorithm="a2c"):
            raise RuntimeError(f"Exploratory checkpoint integrity failed: {manifest_path.parent}")
    evaluation_path = learner_dir / "evaluation.json"
    evaluation = json.loads(evaluation_path.read_text())
    if (evaluation.get("exploratory_only") is not True or evaluation.get("formal_eligible") is not False
            or evaluation.get("benchmark_eligible") is not False or len(evaluation.get("episodes", [])) != 5):
        raise RuntimeError("Exploratory rollout record has invalid provenance or count")
    if any(not row.get("complete") or row.get("ticks") != 72
           or sum(row.get("action_counts", {}).values()) != 72 * 5
           for row in evaluation["episodes"]):
        raise RuntimeError("Each exploratory world must contain complete 72-tick rollouts for five cattle")
    report = {"mode": MODE, "exploratory_only": True, "formal_eligible": False,
              "benchmark_eligible": False, "locked_outcomes_accessed": False,
              "source_commit": EXPECTED_COMMIT, "notebook_sha256": source_notebook_sha,
              "runtime": runtime, "config": cfg, "updates": len(rows),
              "reward_by_update": [row["batch_return"] for row in rows],
              "action_counts_by_update": [row["batch_action_counts"] for row in rows],
              "invalid_actions_by_update": [row["invalid_actions"] for row in rows],
              "checkpoint_generations": [p.parent.name for p in generations],
              "checkpoint_latest": latest.read_text().strip(),
              "policy_export": str(next(learner_dir.glob("torch-a2c-seed-*.npz"))),
              "five_cattle_rollouts": evaluation["episodes"],
              "elapsed_seconds": time.time() - started}
    (ARTIFACT_ROOT / "exploratory-report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print("Exploratory reward trajectory (nonformal):", json.dumps(report["reward_by_update"]))
    print(json.dumps({k: v for k, v in report.items() if k != "five_cattle_rollouts"}, indent=2, sort_keys=True))


In [ ]:
if MODE == "EXPLORATORY_COMPARISON":
    cfg = DEPLOY["exploratory_config"]
    if (cfg != {"task": "exploratory_a2c_variant_comparison", "algorithm": "a2c",
                "variants": ["A", "B", "C", "D"], "episodes_per_variant_seed": 512,
                "seeds": [9483001, 9483002, 9483003], "training_jobs": 12, "evaluation_episodes": 5,
                "comparison_eval_seed": 9483999, "comparison_episodes_per_family": 3,
                "max_training_jobs": 24, "max_devices": 2, "locked_test_access": False,
                "variant_config": {
                    "A": {"reward_variant": "original", "state_reward_scale": 0.1,
                          "normalize_advantage": False, "entropy_coefficient": 0.0, "discount_factor": 1.0},
                    "B": {"reward_variant": "original", "state_reward_scale": 0.1,
                          "normalize_advantage": True, "entropy_coefficient": 0.01, "discount_factor": 1.0},
                    "C": {"reward_variant": "state_aware", "state_reward_scale": 0.1,
                          "normalize_advantage": False, "entropy_coefficient": 0.0, "discount_factor": 1.0},
                    "D": {"reward_variant": "state_aware", "state_reward_scale": 0.1,
                          "normalize_advantage": True, "entropy_coefficient": 0.01, "discount_factor": 1.0}},
                "formal_eligible": False, "benchmark_eligible": False,
                "checkpoint_promotion": False}):
        raise RuntimeError("Exploratory comparison config differs from its fixed allowlist")
    if cfg["training_jobs"] > cfg["max_training_jobs"] or cfg["max_training_jobs"] != 24:
        raise RuntimeError("Exploratory comparison exceeds the 24-job safety limit")
    comparison_root = ARTIFACT_ROOT / "exploratory-comparison"
    comparison_root.mkdir()
    profile_root = comparison_root / "calibration"
    profile_root.mkdir()
    # Synthetic MLP calibration is kept distinct from actual simulator-stage timing.
    mlp_profile_path = profile_root / "device-mlp-profile.json"
    run([sys.executable, "-m", "validation.profile_torch_training", "--devices", "cpu,cuda:0,cuda:1",
         "--batches", "128,5760", "--warmups", "3", "--iterations", "5", "--repeats", "3",
         "--output", mlp_profile_path], cwd=REPO_DIR)
    mlp_profile = json.loads(mlp_profile_path.read_text())
    if len(mlp_profile.get("results", [])) != 18 or not mlp_profile.get("cuda_available"):
        raise RuntimeError("CPU/T4 MLP calibration did not report the expected device phases")

    # Real 16-episode (5,760 decision) collection plus forward/backward work;
    # no optimizer.step, checkpoint write, or learning outcome is produced.
    profile_reports = {}
    for device in ("cpu", "cuda:0", "cuda:1"):
        target = profile_root / f"real-step-{device.replace(':', '-')}.json"
        profile_started = time.perf_counter()
        run([sys.executable, "-m", "validation.train_torch_mvp1", "--profile-only",
             "--profile-device", device, "--profile-output", target, "--profile-repeats", "1"], cwd=REPO_DIR)
        report = json.loads(target.read_text())
        if report.get("optimizer_step") is not False or report.get("checkpoint_write") is not False:
            raise RuntimeError("Calibration unexpectedly updated weights or wrote a checkpoint")
        profile_reports[device] = report
        profile_reports[f"{device}:wall_seconds"] = time.perf_counter() - profile_started

    children = []
    dual_started = time.perf_counter()
    for worker_index, gpu_index in enumerate((0, 1)):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
        report_path = profile_root / f"real-step-dual-{worker_index}.json"
        log_path = profile_root / f"real-step-dual-{worker_index}.log"
        log_file = log_path.open("w")
        process = subprocess.Popen(
            [sys.executable, "-m", "validation.train_torch_mvp1", "--profile-only",
             "--profile-device", "cuda:0", "--profile-output", str(report_path), "--profile-repeats", "1"],
            cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        children.append((worker_index, process, log_file, log_path, report_path))
    for worker_index, process, log_file, log_path, report_path in children:
        code = process.wait()
        log_file.close()
        if code:
            raise RuntimeError(f"Concurrent calibration worker {worker_index} failed; see {log_path}")
        profile_reports[f"dual:{worker_index}"] = json.loads(report_path.read_text())
    dual_makespan = time.perf_counter() - dual_started

    single_times = {device: profile_reports[f"{device}:wall_seconds"]
                    for device in ("cpu", "cuda:0", "cuda:1")}
    best_single_device = min(single_times, key=single_times.get)
    best_single_seconds = single_times[best_single_device]
    dual_throughput_ratio = (2.0 / dual_makespan) / (1.0 / best_single_seconds)
    use_dual = dual_throughput_ratio > 1.0
    train_root = comparison_root / "training-suite"
    training_started = time.perf_counter()
    if use_dual:
        training_processes = []
        for worker_index, gpu_index in enumerate((0, 1)):
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
            log_path = comparison_root / f"training-worker-{worker_index}.log"
            log_file = log_path.open("w")
            process = subprocess.Popen(
                [sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-batch",
                 "--algorithm", "a2c", "--device", "cuda", "--batch-shard-index", str(worker_index),
                 "--batch-shard-count", "2", "--output", str(train_root)],
                cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
            training_processes.append((worker_index, process, log_file, log_path))
        for worker_index, process, log_file, log_path in training_processes:
            code = process.wait()
            log_file.close()
            if code:
                raise RuntimeError(f"Training shard {worker_index} failed; see {log_path}")
        lane = "dual-independent"
    else:
        training_device = "cpu" if best_single_device == "cpu" else "cuda"
        train_env = os.environ.copy()
        if training_device == "cuda":
            train_env["CUDA_VISIBLE_DEVICES"] = best_single_device.split(":")[1]
        run([sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-batch",
             "--algorithm", "a2c", "--device", training_device, "--output", train_root],
            cwd=REPO_DIR, env=train_env)
        lane = "single-device"
    training_seconds = time.perf_counter() - training_started

    expected_updates = 512 // 16
    checks = []
    for variant in ("A", "B", "C", "D"):
        for seed in cfg["seeds"]:
            run_root = train_root / "exploratory-comparison" / f"variant-{variant}" / f"seed-{seed}"
            metrics_path = run_root / "metrics.jsonl"
            rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
            if len(rows) != expected_updates or [r.get("update") for r in rows] != list(range(1, expected_updates + 1)):
                raise RuntimeError(f"{lane} {variant}/{seed} did not complete exactly 512 episodes")
            if any("raw_environment_batch_return" not in row or "training_target_mean_return_to_go" not in row
                   for row in rows):
                raise RuntimeError("Comparison log must separate raw rewards from shaped training targets")
            if any(r.get("exploratory_only") is not True or r.get("formal_eligible") is not False
                   or r.get("benchmark_eligible") is not False for r in rows):
                raise RuntimeError("Comparison training log is not marked exploratory-only")
            if any(not {"rollout_seconds", "forward_loss_seconds", "backward_gradient_seconds",
                        "optimizer_seconds", "checkpoint_io_seconds", "batch_wall_seconds"}.issubset(r.get("timing", {}))
                   for r in rows):
                raise RuntimeError("Comparison training log is missing stage timing")
            evaluation = json.loads((run_root / "a2c" / "evaluation.json").read_text())
            if (evaluation.get("exploratory_only") is not True or evaluation.get("formal_eligible") is not False
                    or len(evaluation.get("episodes", [])) != cfg["evaluation_episodes"]):
                raise RuntimeError("Comparison evaluation must be a separate fixed exploratory-only schedule")
            latest = run_root / "a2c" / "LATEST"
            if not latest.is_file():
                raise RuntimeError("Comparison checkpoint recovery pointer is missing")
            from farm_sim.torch_checkpoint import _is_complete_generation
            generations = sorted((run_root / "a2c").glob("generation-*/manifest.json"))
            if len(generations) != expected_updates or any(
                    not _is_complete_generation(item.parent, expected_algorithm="a2c") for item in generations):
                raise RuntimeError("Comparison checkpoint generation integrity/count check failed")
            checks.append({"variant": variant, "seed": seed, "updates": len(rows),
                           "latest": latest.read_text().strip()})
    # Compare the one selected suite on a distinct fixed generated-world root.
    checkpoint_manifest = {"schema_version": 1, "mode": "exploratory_comparison",
                           "formal_eligible": False, "locked_test_access": False, "conditions": {}}
    for variant in ("A", "B", "C", "D"):
        entries = []
        for seed in cfg["seeds"]:
            checkpoint_root = train_root / "exploratory-comparison" / f"variant-{variant}" / f"seed-{seed}" / "a2c"
            generation_name = (checkpoint_root / "LATEST").read_text().strip()
            entries.append({"generation": str((checkpoint_root / generation_name).resolve()),
                            "algorithm": "a2c", "seed": seed})
        checkpoint_manifest["conditions"][variant] = entries
    checkpoint_manifest_path = comparison_root / "checkpoint-manifest.json"
    checkpoint_manifest_path.write_text(json.dumps(checkpoint_manifest, indent=2, sort_keys=True) + "\n")
    evaluation_root = comparison_root / "generated-world-evaluation"
    run([sys.executable, "-m", "validation.compare_exploratory_checkpoints",
         "--checkpoints", checkpoint_manifest_path,
         "--eval-seed", str(cfg["comparison_eval_seed"]),
         "--episodes-per-family", str(cfg["comparison_episodes_per_family"]),
         "--device", "cpu", "--output", evaluation_root], cwd=REPO_DIR)
    evaluation_summary = evaluation_root / "exploratory-comparison.json"
    evaluation_plot = evaluation_root / "exploratory-comparison.svg"
    evaluation_doc = json.loads(evaluation_summary.read_text())
    if (evaluation_doc.get("mode") != "exploratory_comparison"
            or evaluation_doc.get("evaluation_root_seed") != cfg["comparison_eval_seed"]
            or evaluation_doc.get("formal_eligible") is not False
            or evaluation_doc.get("locked_test_access") is not False
            or set(evaluation_doc.get("conditions", {})) != {"A", "B", "C", "D"}
            or not evaluation_plot.is_file()):
        raise RuntimeError("A-D common-world evaluation is missing or not nonformal")
    for condition in ("A", "B", "C", "D"):
        if set(evaluation_doc["conditions"][condition].get("policies", {})) != {
                "trained", "untrained_same_seed", "heuristic_baseline"}:
            raise RuntimeError(f"Evaluation omitted policy baselines for {condition}")
    evaluation_summary_sha256 = hashlib.sha256(evaluation_summary.read_bytes()).hexdigest()
    evaluation_plot_sha256 = hashlib.sha256(evaluation_plot.read_bytes()).hexdigest()

    report = {"mode": MODE, "protocol": "exploratory-comparison-v1", "config": cfg,
              "exploratory_only": True, "formal_eligible": False, "benchmark_eligible": False,
              "locked_outcomes_accessed": False, "source_commit": EXPECTED_COMMIT,
              "notebook_sha256": source_notebook_sha, "runtime": runtime,
              "synthetic_mlp_calibration": str(mlp_profile_path),
              "real_step_calibration": {key: value for key, value in profile_reports.items()},
              "calibration_devices": single_times, "dual_independent_makespan_seconds": dual_makespan,
              "dual_over_single_throughput_ratio_diagnostic_only": dual_throughput_ratio,
              "selected_schedule": lane, "selected_single_device": best_single_device,
              "comparison_eval_seed": cfg["comparison_eval_seed"],
              "evaluation_summary": str(evaluation_summary),
              "evaluation_summary_sha256": evaluation_summary_sha256,
              "evaluation_plot": str(evaluation_plot), "evaluation_plot_sha256": evaluation_plot_sha256,
              "training_job_guard": 24, "training_jobs": cfg["training_jobs"],
              "completed_jobs": len(checks), "training_seconds": training_seconds,
              "job_checks": checks,
              "claim_boundary": "One nonformal 12-job suite only; calibration has no learning update. Timing is descriptive, not a GPU speedup claim."}
    (comparison_root / "exploratory-comparison-report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


In [ ]:
if MODE == "EXPLORATORY_ITERATION_2":
    cfg = DEPLOY["exploratory_config"]
    expected = {"task": "exploratory_a2c_reward_scale_followup", "algorithm": "a2c",
        "variants": ["E", "F"], "episodes_per_variant_seed": 512,
        "seeds": [9483004, 9483005, 9483006], "training_jobs": 6,
        "comparison_eval_seed": 9483999, "comparison_episodes_per_family": 3,
        "max_training_jobs": 24, "cumulative_jobs_before": 12,
        "max_devices": 2, "locked_test_access": False,
        "variant_config": {
            "E": {"reward_variant": "state_aware", "state_reward_scale": 0.5,
                  "normalize_advantage": False, "entropy_coefficient": 0.0, "discount_factor": 1.0},
            "F": {"reward_variant": "state_aware", "state_reward_scale": 0.5,
                  "normalize_advantage": True, "entropy_coefficient": 0.01, "discount_factor": 1.0}},
        "formal_eligible": False, "benchmark_eligible": False, "checkpoint_promotion": False}
    if cfg != expected or cfg["cumulative_jobs_before"] + cfg["training_jobs"] > cfg["max_training_jobs"]:
        raise RuntimeError("Exploratory follow-up differs from its fixed six-job, cumulative 18/24 allowlist")
    follow_root = ARTIFACT_ROOT / "exploratory-iteration-2"
    train_root = follow_root / "training-suite"
    train_root.mkdir(parents=True)
    # Batch-one real-rollout calibration selected two independent T4 workers;
    # keep the same measured schedule for this bounded follow-up, no DDP.
    children = []
    for worker_index, gpu_index in enumerate((0, 1)):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
        log_path = follow_root / f"training-worker-{worker_index}.log"
        log_file = log_path.open("w")
        process = subprocess.Popen(
            [sys.executable, "-m", "validation.train_torch_mvp1", "--exploratory-iteration-2",
             "--algorithm", "a2c", "--device", "cuda", "--batch-shard-index", str(worker_index),
             "--batch-shard-count", "2", "--output", str(train_root)],
            cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        children.append((worker_index, process, log_file, log_path))
    shard_reports = []
    for worker_index, process, log_file, log_path in children:
        code = process.wait()
        log_file.close()
        if code:
            raise RuntimeError(f"Iteration-2 training shard {worker_index} failed; inspect {log_path}")
        shard_path = train_root / "exploratory-iteration-2" / f"iteration-2-shard-{worker_index}-of-2.json"
        report = json.loads(shard_path.read_text())
        if len(report.get("jobs", [])) != 3:
            raise RuntimeError(f"Iteration-2 shard {worker_index} did not finish exactly three jobs")
        shard_reports.append(report)
    checkpoint_entries = []
    for variant in cfg["variants"]:
        for seed in cfg["seeds"]:
            run_root = train_root / "exploratory-iteration-2" / f"variant-{variant}" / f"seed-{seed}"
            metrics = [json.loads(line) for line in (run_root / "metrics.jsonl").read_text().splitlines() if line.strip()]
            if len(metrics) != 32 or [row.get("update") for row in metrics] != list(range(1, 33)):
                raise RuntimeError(f"Iteration-2 {variant}/{seed} did not complete 512 episodes")
            if any(row.get("formal_eligible") is not False or row.get("exploratory_only") is not True
                   or row.get("benchmark_eligible") is not False for row in metrics):
                raise RuntimeError("Iteration-2 metrics contain unsafe eligibility flags")
            ckpt = run_root / "a2c"
            from farm_sim.torch_checkpoint import _is_complete_generation
            latest = ckpt / (ckpt / "LATEST").read_text().strip()
            if not _is_complete_generation(latest, expected_algorithm="a2c"):
                raise RuntimeError(f"Iteration-2 checkpoint failed integrity check: {latest}")
            checkpoint_entries.append({"variant": variant, "seed": seed, "generation": str(latest)})
    # Evaluate E/F on the same original-reward, held-out world schedule as A-D.
    # These descriptive reports are not formal split access or promotion evidence.
    evaluation_root = follow_root / "generated-world-evaluation"
    evaluation_root.mkdir()
    reports = []
    for entry in checkpoint_entries:
        report_path = evaluation_root / entry["variant"] / f"seed-{entry['seed']}-evaluation.json"
        run([sys.executable, "-m", "validation.evaluate_torch_mvp1",
             "--generation", entry["generation"], "--algorithm", "a2c", "--seed", str(entry["seed"]),
             "--eval-seed", str(cfg["comparison_eval_seed"]),
             "--episodes-per-family", str(cfg["comparison_episodes_per_family"]),
             "--device", "cpu", "--output", report_path], cwd=REPO_DIR)
        report = json.loads(report_path.read_text())
        if (report.get("seed_split", {}).get("evaluation_root_seed") != cfg["comparison_eval_seed"]
                or report.get("locked_pre_outcome_binding") is not None
                or report.get("training_validation_test_disjoint_check", {}).get("training_vs_eval_collision_count") != 0):
            raise RuntimeError("Iteration-2 evaluation was not the expected disjoint exploratory evaluation")
        reports.append({"variant": entry["variant"], "seed": entry["seed"],
                        "report": str(report_path), "report_sha256": hashlib.sha256(report_path.read_bytes()).hexdigest()})
    iteration_report = {"schema_version": 1, "mode": MODE, "formal_eligible": False,
        "benchmark_eligible": False, "locked_outcomes_accessed": False,
        "source_commit": EXPECTED_COMMIT, "runtime": runtime,
        "prior_batch_source_commit": "d2fdecf1a49548056775ad0fecaff4aefc6542e3",
        "prior_batch_jobs": 12, "this_batch_jobs": 6, "cumulative_jobs": 18,
        "resource_scale_change_rationale": "Batch one produced zero forage successes and mean final hunger >0.97 for A-D; this five-fold state-aware shaping scale is a bounded exploratory response.",
        "training_shards": shard_reports, "checkpoints": checkpoint_entries,
        "common_world_evaluations": reports, "comparison_eval_seed": cfg["comparison_eval_seed"]}
    (follow_root / "iteration-2-report.json").write_text(json.dumps(iteration_report, indent=2, sort_keys=True) + "\n")
    print(json.dumps({"mode": MODE, "completed_jobs": len(checkpoint_entries),
                      "evaluation_reports": len(reports), "formal_eligible": False,
                      "locked_outcomes_accessed": False}, indent=2))


In [ ]:
if MODE in {"AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"}:
    from scripts.kaggle_release import AUTORESEARCH_BATCH_CONFIGS, authorize_mode
    from validation.run_autoresearch_training import run_batch, create_candidate_manifest

    cfg = DEPLOY.get("exploratory_config")
    if cfg != AUTORESEARCH_BATCH_CONFIGS.get(MODE):
        raise RuntimeError("Autoresearch config differs from the exact allowlisted batch")
    authorize_mode(MODE, {"release_manifest_sha256": DEPLOY.get("release_manifest_sha256")}, config=cfg)
    if cfg.get("max_additional_training_jobs") != 24 or cfg.get("training_jobs") != 6:
        raise RuntimeError("Autoresearch cohort exceeds its six-run batch / 24-additional-job bound")
    if cfg.get("locked_test_access") is not False or cfg.get("formal_eligible") is not False:
        raise RuntimeError("Autoresearch must remain nonformal and deny locked outcomes")

    auto_root = ARTIFACT_ROOT / "autoresearch" / f"batch-{cfg['batch']}"
    calibration_root = auto_root / "calibration"
    calibration_root.mkdir(parents=True)
    # Bounded 16-episode collect+forward/backward probes only; no optimizer step,
    # no checkpoint, and no training outcome. Profile all usable single devices.
    profile_reports = {}
    for device in ("cpu", "cuda:0", "cuda:1"):
        target = calibration_root / f"profile-{device.replace(':', '-')}.json"
        started = time.perf_counter()
        run([sys.executable, "-m", "validation.train_torch_mvp1", "--profile-only",
             "--profile-device", device, "--profile-output", target, "--profile-repeats", "1"], cwd=REPO_DIR)
        report = json.loads(target.read_text())
        if report.get("optimizer_step") is not False or report.get("checkpoint_write") is not False:
            raise RuntimeError("Autoresearch calibration unexpectedly learned or checkpointed")
        profile_reports[device] = {"elapsed_wall_seconds": time.perf_counter() - started, **report}
    best_single_device = min(profile_reports, key=lambda device: profile_reports[device]["elapsed_wall_seconds"])
    best_single_seconds = profile_reports[best_single_device]["elapsed_wall_seconds"]

    # Measure independent T4 workers concurrently; select them only if the
    # two-worker makespan yields greater measured 16-episode throughput.
    children = []
    dual_started = time.perf_counter()
    for worker, gpu_index in enumerate((0, 1)):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
        report_path = calibration_root / f"profile-dual-{worker}.json"
        log_path = calibration_root / f"profile-dual-{worker}.log"
        log_file = log_path.open("w")
        process = subprocess.Popen(
            [sys.executable, "-m", "validation.train_torch_mvp1", "--profile-only",
             "--profile-device", "cuda:0", "--profile-output", str(report_path), "--profile-repeats", "1"],
            cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
        children.append((worker, process, log_file, log_path, report_path))
    for worker, process, log_file, log_path, report_path in children:
        code = process.wait()
        log_file.close()
        if code:
            raise RuntimeError(f"Dual-device profile worker {worker} failed; inspect {log_path}")
        profile_reports[f"dual-{worker}"] = json.loads(report_path.read_text())
    dual_makespan = time.perf_counter() - dual_started
    dual_throughput_ratio = (2.0 * best_single_seconds) / dual_makespan
    use_dual = dual_throughput_ratio > 1.0

    training_root = auto_root / "training"
    training_started = time.perf_counter()
    if use_dual:
        training_children = []
        for worker, gpu_index in enumerate((0, 1)):
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
            log_path = auto_root / f"training-worker-{worker}.log"
            log_file = log_path.open("w")
            process = subprocess.Popen(
                [sys.executable, "-m", "validation.run_autoresearch_training",
                 "--batch", str(cfg["batch"]), "--output", str(training_root),
                 "--device", "cuda", "--shard-index", str(worker), "--shard-count", "2"],
                cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
            training_children.append((worker, process, log_file, log_path))
        for worker, process, log_file, log_path in training_children:
            code = process.wait()
            log_file.close()
            if code:
                raise RuntimeError(f"Autoresearch GPU training shard {worker} failed; inspect {log_path}")
        selected_schedule = "two-independent-T4-jobs"
        training_device = "cuda"
    else:
        training_device = "cpu" if best_single_device == "cpu" else "cuda"
        env = os.environ.copy()
        if training_device == "cuda":
            env["CUDA_VISIBLE_DEVICES"] = best_single_device.split(":")[1]
        run([sys.executable, "-m", "validation.run_autoresearch_training",
             "--batch", str(cfg["batch"]), "--output", training_root,
             "--device", training_device, "--shard-count", "1"], cwd=REPO_DIR, env=env)
        selected_schedule = f"single-{training_device}"
    training_seconds = time.perf_counter() - training_started

    # Verify all six new training attempts, all per-update checkpoints, all
    # inference exports, and nonformal markers before artifact export.
    checks = []
    for variant in cfg["variants"]:
        for seed in cfg["seeds"]:
            run_root = training_root / f"variant-{variant}" / f"seed-{seed}"
            metrics_path = run_root / "metrics.jsonl"
            rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
            if len(rows) != 32 or [row.get("update") for row in rows] != list(range(1, 33)):
                raise RuntimeError(f"Autoresearch {variant}/{seed} did not complete 512 episodes")
            if any(row.get("exploratory_only") is not True or row.get("formal_eligible") is not False
                   or row.get("benchmark_eligible") is not False or row.get("invalid_actions") != 0 for row in rows):
                raise RuntimeError("Autoresearch training metrics violate nonformal/finite output guards")
            checkpoint = run_root / "a2c"
            latest = checkpoint / (checkpoint / "LATEST").read_text().strip()
            from farm_sim.torch_checkpoint import _is_complete_generation
            generations = sorted(checkpoint.glob("generation-*/manifest.json"))
            if len(generations) != 32 or not _is_complete_generation(latest, expected_algorithm="a2c"):
                raise RuntimeError(f"Autoresearch checkpoint generations failed validation: {checkpoint}")
            policy = checkpoint / f"torch-a2c-seed-{seed}.npz"
            if not policy.is_file() or not policy.with_suffix(".json").is_file():
                raise RuntimeError(f"Autoresearch inference export missing: {policy}")
            checks.append({"variant": variant, "seed": seed, "updates": len(rows),
                           "latest_generation": latest.name, "inference_export": str(policy),
                           "inference_sha256": hashlib.sha256(policy.read_bytes()).hexdigest()})
    candidate_manifest = create_candidate_manifest(batch=cfg["batch"], output=training_root)
    if len(json.loads(candidate_manifest.read_text()).get("candidate_checkpoints", [])) != 6:
        raise RuntimeError("Hash-bound candidate manifest did not contain all six policy exports")
    report = {"schema_version": 1, "mode": MODE, "protocol": cfg["protocol"],
        "config": cfg, "source_commit": EXPECTED_COMMIT, "notebook_sha256": source_notebook_sha,
        "runtime": runtime, "cuda_forward_backward": "PASS", "gpu_names": runtime["gpu_names"],
        "calibration": profile_reports, "dual_gpu_makespan_seconds": dual_makespan,
        "dual_vs_best_single_throughput_ratio": dual_throughput_ratio,
        "selected_schedule": selected_schedule, "selected_single_device": best_single_device,
        "training_wall_seconds": training_seconds, "jobs_started": 6,
        "jobs_completed": len(checks), "max_additional_training_jobs": 24,
        "cumulative_jobs_before": cfg["cumulative_jobs_before"],
        "cumulative_jobs_after": cfg["cumulative_jobs_before"] + len(checks),
        "candidate_manifest": str(candidate_manifest),
        "candidate_manifest_sha256": hashlib.sha256(candidate_manifest.read_bytes()).hexdigest(),
        "job_checks": checks, "formal_eligible": False, "benchmark_eligible": False,
        "locked_outcomes_accessed": False,
        "claim_boundary": "Kaggle exploratory training only; evaluation/acceptance is run locally on new shared worlds using unchanged original reward. No biological validation or GPU speedup claim."}
    (auto_root / "autoresearch-training-report.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2, sort_keys=True))


## Integrity-checked artifact export

Autoresearch export includes nonformal learning curves, action/resource diagnostics, five-cattle checkpoints, NumPy policy exports, all checkpoint generations, runtime calibration, and run ledgers. Artifacts are checksummed and remain ineligible for formal analysis, benchmark comparison, or locked-data evaluation. Acceptance is evaluated separately on new common worlds with the unchanged simulator reward.


In [ ]:
if MODE in {"READINESS", "SMOKE", "EXPLORATORY", "EXPLORATORY_COMPARISON", "EXPLORATORY_ITERATION_2", "AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"}:
    files = []
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file() and path.name not in {"artifact-manifest.json", "pasture-readiness.zip"}:
            files.append({"path": path.relative_to(ARTIFACT_ROOT).as_posix(),
                          "size_bytes": path.stat().st_size,
                          "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    artifact_manifest = {"schema_version": 1, "mode": MODE,
        "run_id": RUN_ID, "source_commit": EXPECTED_COMMIT,
        "release_manifest_sha256": (DEPLOY.get("release_manifest_sha256")
            if MODE in {"READINESS", "EXPLORATORY", "EXPLORATORY_COMPARISON", "EXPLORATORY_ITERATION_2", "AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"} else None),
        "notebook_sha256": source_notebook_sha, "runtime": runtime,
        "benchmark_eligible": False, "formal_eligible": False,
        "exploratory_only": MODE in {"EXPLORATORY", "EXPLORATORY_COMPARISON", "EXPLORATORY_ITERATION_2", "AUTORESEARCH_BATCH_1", "AUTORESEARCH_BATCH_2", "AUTORESEARCH_BATCH_3"}, "locked_outcomes_accessed": False,
        "locked_test_access": False,
        "files": files}
    manifest_path = ARTIFACT_ROOT / "artifact-manifest.json"
    manifest_path.write_text(json.dumps(artifact_manifest, indent=2, sort_keys=True) + "\n")
    archive_path = ARTIFACT_ROOT / "pasture-artifacts.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(manifest_path, "artifact-manifest.json")
        for entry in files: archive.write(ARTIFACT_ROOT / entry["path"], entry["path"])
    with zipfile.ZipFile(archive_path) as archive:
        bad = archive.testzip()
        if bad: raise RuntimeError(f"ZIP integrity failed at {bad}")
    print("Artifact directory:", ARTIFACT_ROOT)
    print("Artifact manifest entries:", len(files))
    print("ZIP SHA-256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())
